In [7]:
# load up USPTO-50k test dataset
import pandas as pd

df = pd.read_csv("/kaggle/working/DFS/DeepRetro/data/uspto_50k_test_250.csv")
df.head()

,input,output,reaction_type,cluster_id
0,CS(=O)c1cccc(-c2nc(C=O)ccc2OCCO[Si](C)(C)C(C)(...,CC(C)(C)[Si](C)(C)OCCOc1ccc(C=O)nc1Br.CS(=O)c1...,3,0
1,CC(C)=CCSc1ccc(Br)cc1,CC(C)=CCBr.Sc1ccc(Br)cc1,1,0
2,CCC1(c2ccc(C=O)s2)OCCO1,CCC1(c2cccs2)OCCO1.CN(C)C=O,3,0
3,O=C1Nc2ccccc2C1c1cc(Br)ccc1O,O=C1Nc2ccccc2C1(O)c1cc(Br)ccc1O,9,0
4,CCCCCCCCCCCCCCCCOCC(CN)CC#N,CCCCCCCCCCCCCCCCOCC(CC#N)CN=[N+]=[N-],9,0


In [9]:
mol1 = df.iloc[0]['input']
trg1 = df.iloc[0]['output']

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
Olmo3_Instruct = AutoModelForCausalLM.from_pretrained('allenai/Olmo-3-7B-Instruct',
                             device_map = 'auto',
                             dtype='float16',
                             low_cpu_mem_usage=True)

Loading weights:   0%|          | 0/355 [00:00<?, ?it/s]

In [ ]:
USER_PROMPT = """You are an expert organic chemist specializing in retrosynthesis. When given a target molecule, you will perform a single-step retrosynthesis, providing 3-5 possible precursor molecules or reactions that could lead to the formation of the target molecule. 

Present your final analysis in a specific JSON format. For each suggestion, provide the precursor molecules in SMILES notation and a brief explanation of the reaction type and any key conditions or reagents needed. Use standard organic chemistry notation and terminology in your explanations. 

If the molecule is too simple for meaningful retrosynthesis, state this in a single JSON object with an appropriate explanation.

Perform a single-step retrosynthesis on the following molecule, providing 3-5 possible precursors or reactions:



Present your final analysis in the following JSON format:

<json>
{
  "data": [
    [precursor1_SMILES, precursor2_SMILES, ...],
    [precursor1_SMILES, precursor2_SMILES, ...],
    ...
  ],
  "explanation": [
    "explanation 1",
    "explanation 2",
    ...
  ],
  "confidence_scores": [
    confidence_score1,
    confidence_score2,
    ...
  ]
}
</json>

For each suggestion in the "data" array, provide the precursor molecules in SMILES notation. Ensure to provide only valid SMILES strings.

In the corresponding "explanation" array, briefly explain the reaction type and any key conditions or reagents needed.

In the "confidence_scores" array, provide a confidence score for each suggestion between 0 and 1, indicating your confidence in the proposed retrosynthesis pathway.

Ensure that the number of entries in "data", "explanation", and "confidence_scores" are the same.
"""

In [12]:
USER_PROMPT

'You are an expert organic chemist specializing in retrosynthesis. When given a target molecule, you will perform a single-step retrosynthesis, providing 3-5 possible precursor molecules or reactions that could lead to the formation of the target molecule. \n\nPresent your final analysis in a specific JSON format. For each suggestion, provide the precursor molecules in SMILES notation and a brief explanation of the reaction type and any key conditions or reagents needed. Use standard organic chemistry notation and terminology in your explanations. \n\nIf the molecule is too simple for meaningful retrosynthesis, state this in a single JSON object with an appropriate explanation.\n\nPerform a single-step retrosynthesis on the following molecule, providing 3-5 possible precursors or reactions:\n\n{mol1}\n\nPresent your final analysis in the following JSON format:\n\n<json>\n{\n  "data": [\n    [precursor1_SMILES, precursor2_SMILES, ...],\n    [precursor1_SMILES, precursor2_SMILES, ...],\n